----
## Apprentissage par renforcement - Q_learning - v3.2
----
The purpose of this new version is to fit the achitecture of the qtable of Johannes meaning that state is represented by a list `[x, x, x, x, x, x, x, x, x]` of lenth $9$

In [37]:
import random
import numpy as np
from copy import deepcopy
from tqdm import tqdm

In [38]:
# Board structure
# | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |

# This dictiionary provides the possible actions from each postion of the table.
CONSTRAINT_DICT = {
    0: [1, 3, 4],
    1: [0, 2, 4],
    2: [1, 4, 5],
    3: [0, 4, 6],
    4: [0, 1, 2, 3, 5, 6, 7, 8],
    5: [2, 4, 8],
    6: [3, 4, 7],
    7: [4, 6, 8],
    8: [4, 5, 7], 
}

In [39]:
def state_to_tuple(state):
    """
    Convert the state into Tuple so that we could use it as a key of a dictionnary
    """
    return tuple(state)

In [40]:
def available_action(state = np.zeros((9,)), is_current_player_agent=True, agent_symbol=1, human_symbol=-1) :
    """
    This function provides available actions depending on the board and the case (deployment vs moving)
    - In deployment state : return a list of available position
    - In Moving state : returns a list of available moving actions of the current player
    """
    symbol_player = agent_symbol if is_current_player_agent else human_symbol
    state = np.array(state)
    # Here we check if we reach the maximum number of pawn ('pion' in french) of the current player
    nb_pion = sum([1 for element in state if element == symbol_player])

    free_positions = []
    current_player_possible_moves = []
    
    if nb_pion != 3:  # Case of deployement
        free_positions = [idx for (idx, pos) in enumerate(state) if pos not in [agent_symbol, human_symbol]] 
        return free_positions

    else :  # Case of moving
        # Find position of the master pawn
        pawn_positions = [idx for (idx, pos) in enumerate(state) if pos == symbol_player]

        for pawn_pos in pawn_positions : 
            current_player_possible_moves += [(pawn_pos, pos) for pos in CONSTRAINT_DICT[pawn_pos] if int(state[pos]) not in [agent_symbol, human_symbol]]
        return current_player_possible_moves


In [41]:
def get_new_state(state, action, is_current_player_agent=True, agent_symbol=1, human_symbol=-1):
    """
    Deployement: action is an integer
    Moving: action is a tuple (from,to)
    return the new state of the board according to the action and the player
    
    """
    player_symbol = agent_symbol if is_current_player_agent else human_symbol
    new_state = deepcopy(np.array(state))

    # Deployment state
    if isinstance(action, int):
        new_state[action] = player_symbol
    # Moving state
    elif isinstance(action, tuple):
        new_state[action[0]] = 0
        new_state[action[1]] = player_symbol
    else:
        return None
    return new_state

### Temporal Difference : 
### $$ TD (s_t, a_t) = r_t + \gamma \max Q(s_{t+1}, a) - Q(s_t, a_t)$$  

### Bellman Equation  
### $$ Q^{new}(s_t, a_t) = Q^{old}(s_t, a_t) + \alpha TD (s_t, a_t) $$ 

In [43]:
def update_qtable(q_table, state, action, reward, new_state, alpha=0.1, gamma = 0.9, agent_symbol=1, human_symbol=-1):
    """
    Update the qtable by using temporal difference and the Bellman equation
    - alpha : is the learning rate
    - gamma : is the reward factor
    """
    DEBUG = False
    state = state_to_tuple(state)
    new_state = state_to_tuple(new_state)
    # add state if missing in the q_table
    if state not in q_table:
        if DEBUG == True : print("WARNING : state not in q_table : ") , show(state)
        q_table[state] = {_action : 0 for _action in available_action(state, True, agent_symbol, human_symbol)} # dans available_action is_current_player_agent=True par défaut non? Pourquoi encore True  ici ? 

    # add action if missing
    if action not in q_table[state]:
        if DEBUG == True : print("WARNING : action not in state: ", action)
        q_table[state][action] = 0

    # add new state if missing
    if new_state not in q_table:
        if DEBUG == True : print("WARNING : new_state not in q_table : ") , show(new_state)
        q_table[new_state] = {_action : 0 for _action in available_action(new_state, True, agent_symbol, human_symbol)}

    q_value_max = max(q_table[new_state].values(), default=0)
    q_table[state][action] += alpha * (reward + gamma * q_value_max - q_table[state][action])

In [44]:
def choose_action(state, q_table, epsilon = 0.1, is_current_player_agent=True, debug=False, agent_symbol=1, human_symbol=-1):
    """
    Choose epsilon*100% random action and ((1-epsilon)*100% ) best action
    - For exploration we recommand epsilon = 0.9
    - For exploitation we recommand epsilon = 0.1
    """
    actions = available_action(state, is_current_player_agent=is_current_player_agent, agent_symbol=agent_symbol, human_symbol=human_symbol)
    #print("ChooseAction::Available : ", actions)
    if random.uniform(0, 1) < epsilon: # Choose random action
        return random.choice(actions)
    else:  # Choose best action in the qtable according to current state
        state = state_to_tuple(state)
        if state not in q_table:
            #if debug:
                #print(f"Warning : STATE {state} NOT IN Q_TABLE --> RandomAction")
            q_table[state] = {_action : 0 for _action in available_action(state, is_current_player_agent=is_current_player_agent, agent_symbol=agent_symbol, human_symbol=human_symbol)}
        # print("qtable: ", q_table[state])   # DreyTest
        #return max(q_table[state], key=q_table[state].get) ## Argmax policy

        ## Let's try the delta approach
            best_states = []
            
        

In [181]:
## TEST ZONE
state = [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
action = choose_action(state,q_table2)
print(action)


0


In [45]:
def show(state):
    for i in [0,3,6]:
        print(f'|\t{state[i]}\t|\t{state[i+1]}\t|\t{state[i+2]}\t|')
    print()

In [46]:
def train(q_table=dict(), q_table_adv=None, episode=10000, agent_symbol=1, human_symbol=-1, epsilon = 0.9, alpha=0.1, gamma = 0.9, debug=False):
    """
    - For exploration we recommand epsilon = 0.9
    - For exploitation we recommand epsilon = 0.1
    """
    is_agent_play_first = True
    for nb_episode in tqdm(range(episode)):
        state = np.zeros((9,))  # Empty Board
        if debug: print("New Round +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++" )   # DreyTest

        if is_agent_play_first and (nb_episode >= (episode / 2)):  # Take advantage of 'Lazy evaluation'
            is_agent_play_first = False
            
        while not game_ended(state, agent_symbol, human_symbol)[0]:
            
            if is_agent_play_first :
                old_state = deepcopy(state)
                action = choose_action(state, q_table, epsilon, agent_symbol=agent_symbol, human_symbol=human_symbol)
                if debug: print("action_ia: ", action)   # DreyTest
                while not action in available_action(state, is_current_player_agent=True, agent_symbol=agent_symbol, human_symbol=human_symbol):
                    action = choose_action(state, q_table, epsilon, agent_symbol=agent_symbol, human_symbol=human_symbol)
                state = get_new_state(state, action, is_current_player_agent=True, agent_symbol=agent_symbol, human_symbol=human_symbol)    # Agent play
                new_state = deepcopy(state)
                if debug: print("IA play"), show(state)   # DreyTest
                
                
            if is_agent_play_first and ((True, agent_symbol) == game_ended(state, agent_symbol=agent_symbol, human_symbol=human_symbol)): # cet if est avec le else ci-dessous
                reward = 60
                if debug: print("Agent win"), show(state)   # DreyTest
                
            else: # cet else n'est pas celui du premier if, mais plutôt du second juste au dessus
                is_agent_play_first = True # sans ceci, après la moitié de l'episode quand l'adversaire jouera en premier, l'agent ne pourrait plus jouer : l'adversaire jouera indefinitivement
                if debug: print("Available : ", available_action(state, is_current_player_agent=False, agent_symbol=agent_symbol, human_symbol=human_symbol))
                if q_table_adv == None:
                    action_adv = random.choice(available_action(state, is_current_player_agent=False, agent_symbol=agent_symbol, human_symbol=human_symbol))
                else :
                    action_adv = choose_action(state, q_table_adv, epsilon=0.5, is_current_player_agent=False, debug=True, agent_symbol=agent_symbol, human_symbol=human_symbol)
                    if debug: print("action_adv: ", action_adv)   # DreyTest
                    while not action_adv in available_action(state, is_current_player_agent=False, agent_symbol=agent_symbol, human_symbol=human_symbol):
                        action_adv = choose_action(state, q_table_adv, epsilon=0.5, agent_symbol=agent_symbol, human_symbol=human_symbol,is_current_player_agent=False)    
                    state = get_new_state(state, action_adv, is_current_player_agent=False, agent_symbol=agent_symbol, human_symbol=human_symbol)    # Human play (precisely opponent)

                if (True, human_symbol) == game_ended(state, agent_symbol, human_symbol):
                    reward = -50
                    if debug: print("opponent win"), show(state)  # DreyTest
                    
                else:
                    reward = 0
                    if debug: print("opponent plays"), show(state)  # DreyTest
                    

            update_qtable(q_table, old_state, action, reward, new_state, alpha, gamma, agent_symbol, human_symbol)
    return q_table

In [47]:
def game_ended(state = np.zeros((9,)), agent_symbol=1, human_symbol=-1):
    """
    OUTPUT:
    - (True, winner_symbol) : if someone win
    - (False, 0) : otherwise
    """
    # Convert state into a matrix: 3x3
    state = np.reshape(state, (3,3))
    
    # Compute sums by row, colonn and diagonal
    sum_by_row, sum_by_col = state.sum(axis=1), state.sum(axis=0)
    sum_by_diags = np.array((np.sum([state[i, i] for i in range(3)]), np.sum([state[i, 2-i] for i in range(3)])))
    # Cheking if we have a winner
    mask_agent = (np.hstack((sum_by_row, sum_by_col, sum_by_diags)) == (3 * agent_symbol))
    mask_human = (np.hstack((sum_by_row, sum_by_col, sum_by_diags)) == (3 * human_symbol))
    if mask_agent.any():
        return True, agent_symbol
    if mask_human.any():
        return True, human_symbol
    return False, 0

In [48]:
q_table_adv = train(episode=8000, epsilon=0.9, agent_symbol=-1, human_symbol=1, debug =False)

100%|█████████████████████████████████████████████████████████████████████████████| 8000/8000 [00:21<00:00, 366.70it/s]


In [49]:
# Save the q_table for future uses
np.save("q_table_test_test", q_table_adv)

In [50]:
q_table2 = train(q_table=dict(), q_table_adv=q_table_adv, episode=8000)

100%|█████████████████████████████████████████████████████████████████████████████| 8000/8000 [00:15<00:00, 531.24it/s]


In [51]:
len(q_table_adv), len(q_table2)

(2902, 4400)

In [52]:
np.save("q_table_test_test_best", q_table2)

In [147]:
def play(q_table,is_agent_playing_first=True,agent_symbol=1, human_symbol=-1,epsilon=0):
    state = np.zeros((9,))
    
    while not game_ended(state)[0] :
        print(game_ended(state)[0])
        if is_agent_playing_first:
            action = choose_action(state, q_table, epsilon = epsilon, debug=False)
            state = get_new_state(state, action, is_current_player_agent=True, agent_symbol=1, human_symbol=-1)
            show(state)
        if is_agent_playing_first and ((True, agent_symbol) == game_ended(state)):
            print("Agent win"), show(state)   # Test
                
        else:
            if is_agent_playing_first == False:
                is_agent_playing_first = True
            actions = available_action(state, is_current_player_agent=False)
            print('Your turn to move a pawn...')
            print('Your possible actions are:', actions)
            action = input("Make a choice: ")
            if isinstance(actions[0], int):
                action_adv = int(action)
            else:  # tuple
                action_adv = tuple(int(x) for x in action.split())
            
            state = get_new_state(state, action_adv, is_current_player_agent=False, agent_symbol=1, human_symbol=-1)
            show(state)
            if (True, human_symbol) == game_ended(state):
                print("opponent win"), show(state)

In [177]:

play(q_table2)



False
|	1.0	|	0.0	|	0.0	|
|	0.0	|	0.0	|	0.0	|
|	0.0	|	0.0	|	0.0	|

Your turn to move a pawn...
Your possible actions are: [1, 2, 3, 4, 5, 6, 7, 8]


Make a choice:  


ValueError: invalid literal for int() with base 10: ''

In [185]:
q_table2

{(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0): {0: 0.0,
  1: 0.0,
  2: 0.0,
  3: 0.0,
  4: 0.0,
  5: 0.0,
  6: 0.0,
  7: 0.0,
  8: 0.0},
 (0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0): {0: 0,
  1: 0,
  2: 0,
  3: 0,
  4: 0,
  5: 0,
  7: 0,
  8: 0},
 (-1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0): {1: 0.0,
  2: 0.0,
  3: 0.0,
  4: 0.0,
  5: 0.0,
  7: 0.0,
  8: 0.0},
 (-1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0): {1: 0,
  2: 0,
  3: 0,
  4: 0,
  5: 0,
  8: 0},
 (-1.0, -1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0): {2: 0.0,
  3: -20.14745,
  4: -11.09745,
  5: -26.12897555,
  8: 34.1719674},
 (-1.0, -1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0): {(3, 4): 0,
  (6, 4): 0,
  (7, 4): 0,
  (7, 8): 0},
 (0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0): {0: 0,
  1: 0,
  3: 0,
  4: 0,
  5: 0,
  6: 0,
  7: 0,
  8: 0},
 (0.0, 0.0, 1.0, 0.0, 0.0, -1.0, 0.0, 0.0, 0.0): {0: 0.0,
  1: 0.0,
  3: 0.0,
  4: 0.0,
  6: 0.0,
  7: 0.0,
  8: 0.0},
 (0.0, 0.0, 1.0, 0.0, 0.0, -1.0, 0.0, 1.0, 0.0): {0: 0,
  1: 0,
  3: 0,
 